# Consolidação: PaSST e HTS-AT congelados

Este notebook consulta a etapa consolidada e permite repetir o protocolo **somente com ativação explícita**. Não há fine-tuning. A execução original foi realizada pelo CLI e seus registros são lidos aqui; consultar resultados não inicia treinamento.

A mesma divisão por fonte é preservada. Seleção: validação; confirmação: cinco seeds novas. O teste da rodada inicial já foi consultado, portanto os resultados não são totalmente cegos. DP entre seeds não é DP entre conjuntos de fontes.

In [ ]:
from pathlib import Path
import json, os
ROOT = Path.cwd().resolve()
if not (ROOT/'src').is_dir():
    ROOT = ROOT.parent
assert (ROOT/'configs/linear_probe/consolidation_mono_disc.json').exists()
import sys
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
RUN_CONSOLIDATION = False
PLAN = ROOT/'configs/linear_probe/consolidation_mono_disc.json'
plan = json.loads(PLAN.read_text(encoding='utf-8'))
plan

## Protocolo e execução opcional

Oito candidatos por modelo (batch 128/1.024, scaler de treino opcional e pesos de classe opcionais), duas finalistas avaliadas nas seeds 42/7/21 e cinco seeds novas 101/202/303/404/505. Até 200 épocas, scheduler e parada antecipada por F1 macro de validação.

Todos os encoders e frontends permanecem congelados. Reutilizamos caches auditados; apenas a cabeça linear de 9.997 parâmetros é treinada. O scaler é parte do modelo e precisa ser carregado com a cabeça.

Ativar a variável abaixo cria **uma nova sessão completa**; para ler a sessão concluída, mantenha False. O CLI é a forma documentada para registrar também o log do console.

In [ ]:
if RUN_CONSOLIDATION:
    os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
    import torch
    torch.set_num_threads(4)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.use_deterministic_algorithms(True)
    from src.training.consolidation import run_consolidation
    session, _ = run_consolidation(PLAN, allow_training=True)
else:
    print('Treinamento desativado; consultando resultados existentes.')

## Resultados confirmados

Média e desvio padrão amostral nas mesmas partições. O resumo portátil contém hashes, seleção, seeds, épocas, tempos e métricas por classe. Os dados e checkpoints permanecem locais.

In [ ]:
from IPython.display import display, Markdown, Image
summary_path = ROOT/'results/transfer_learning_consolidated_summary.json'
assert summary_path.exists(), 'Execute primeiro o protocolo e scripts/report_transfer_consolidation.py.'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
session = ROOT/summary['session']
print('Sessão:', summary['session'])
print('Tempo do protocolo (min):', round(summary['elapsed_seconds']/60, 2))
for model, values in summary['models'].items():
    print(model, 'configuração:', values['configuration'])
    for key in ('accuracy', 'macro_f1', 'macro_precision', 'macro_recall'):
        value = values['aggregate'][key]
        print(f'  {key}: {100*value["mean"]:.2f} ± {100*value["std"]:.2f}%')
print('Predições recarregadas reproduzidas:', summary['all_predictions_reproduced'])

In [ ]:
for model in ('passt', 'htsat'):
    display(Markdown('### '+model))
    for name in ('confirmation_curves.png', 'mean_test_confusion.png'):
        path = session/model/name
        if path.exists(): display(Image(filename=str(path)))

## Relatório e inferência

Consulte `results/transfer_learning_consolidated_report.md` e `docs/transfer_learning_consolidation.md`. O resumo escolhe uma cabeça representativa **pela validação**, não pelo teste.

Para inferência futura:
```python
from src.models.consolidated_probe import load_consolidated_probe
from src.data.transfer import EFFECTS
run = ROOT / summary["models"]["htsat"]["representative_run"]
model = load_consolidated_probe(run, device="cuda")
# waveform: tensor [batch, 64000], mono, 32 kHz, escala float do WAV
# logits = model(waveform.to("cuda"))
# classes = [EFFECTS[i] for i in logits.argmax(1).tolist()]
```
A extração original usou encoder CUDA BF16 e cabeça/scaler float32. O loader preserva essa receita. Para comparar os modelos em outra precisão ou em novas fontes, documentar como nova avaliação; não modificar os resultados desta sessão.